In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/dataset_limpio.csv")

df_new = df.copy()

# Simulamos drift cambiando distribución
df_new["capital_prestado"] = df_new["capital_prestado"] * np.random.uniform(1.1, 1.3)

df_new.to_csv("../data/dataset_nuevo.csv", index=False)

In [2]:
import sys 
import os
sys.path.append(os.path.abspath(".."))

In [3]:
from src.model_monitoring import DataDriftMonitor
import pandas as pd

df_ref = pd.read_csv("../data/dataset_limpio.csv")
df_curr = pd.read_csv("../data/dataset_nuevo.csv")

monitor = DataDriftMonitor(df_ref, df_curr)

print("KS:", monitor.ks_test("capital_prestado"))
print("PSI:", monitor.psi("capital_prestado"))
print("JS:", monitor.jensen_shannon("capital_prestado"))

KS: (np.float64(0.10826345048525393), np.float64(1.2924534018158158e-54))
PSI: 0.051853554897401816
JS: 0.07855877561498967


In [2]:
from pathlib import Path
import sys, os

# Buscar hacia arriba hasta encontrar una carpeta "src"
p = Path.cwd().resolve()
root = None
for parent in [p] + list(p.parents):
    if (parent / "src").is_dir():
        root = parent
        break

print("CWD:", p)
print("ROOT detectado:", root)

if root is None:
    raise FileNotFoundError("No encuentro la carpeta 'src' subiendo desde el directorio actual.")

# Poner el root del proyecto en sys.path
sys.path.insert(0, str(root))

# Checks
print("Existe src/model_monitoring.py?:", (Path(root) / "src" / "model_monitoring.py").exists())
print("sys.path[0]:", sys.path[0])
print("Contenido de src:", [x.name for x in (Path(root) / "src").iterdir()])

CWD: C:\Users\USUARIO\OneDrive\Escritorio\ProyectoM5_PalomaMarquez\mlops-dspt01\notebooks
ROOT detectado: C:\Users\USUARIO\OneDrive\Escritorio\ProyectoM5_PalomaMarquez\mlops-dspt01
Existe src/model_monitoring.py?: True
sys.path[0]: C:\Users\USUARIO\OneDrive\Escritorio\ProyectoM5_PalomaMarquez\mlops-dspt01
Contenido de src: ['.gitkeep', 'ft_engineering.py', 'model_monitoring.py', '__pycache__']


In [3]:
from src.model_monitoring import DataDriftMonitor

In [1]:
import streamlit as st
import pandas as pd
from src.model_monitoring import DataDriftMonitor

st.title("Monitoreo de Data Drift")

df_ref = pd.read_csv("../data/dataset_limpio.csv")
df_curr = pd.read_csv("../data/dataset_nuevo.csv")

monitor = DataDriftMonitor(df_ref, df_curr)

column = st.selectbox("Seleccionar variable", df_ref.columns)

ks_stat, p_value = monitor.ks_test(column)
psi_value = monitor.psi(column)
js_value = monitor.jensen_shannon(column)

st.write("### Métricas de Drift")
st.write("KS:", ks_stat)
st.write("p-value:", p_value)
st.write("PSI:", psi_value)
st.write("Jensen-Shannon:", js_value)

if psi_value > 0.2:
    st.error("🚨 Drift significativo detectado")
elif psi_value > 0.1:
    st.warning("⚠️ Drift moderado")
else:
    st.success("✅ Sin drift relevante")

ModuleNotFoundError: No module named 'src'